In [1]:
from gensim.models import KeyedVectors

# Load directly using KeyedVectors
pretrained_w2v = KeyedVectors.load_word2vec_format(
    "/home/osbox/gensim-data/word2vec-google-news-300/word2vec-google-news-300.gz", 
    binary=True
)

print("Loaded successfully from path!")

Loaded successfully from path!


In [14]:
print("Existence of various words in word2vec of google:")
for word in ["lol", "sigma", "skibidi", "ohio", "sus", "illegal"]:
    print(word, ":", word in pretrained_w2v)

Existence of various words in word2vec of google:
lol : True
sigma : True
skibidi : False
ohio : True
sus : True
illegal : True


In [2]:
print("Who's similar to word 'sus'?")
for word, score in pretrained_w2v.most_similar("sigma", topn=3):
    print(word, f"is {score*100:.2f}% similar.")

Who's similar to word 'sus'?


Six_Sigma is 52.46% similar.
DMAIC is 47.96% similar.
receptor_agonism is 45.19% similar.


In [27]:
pretrained_w2v.doesnt_match(["fire", "water", "earth", "pizza"])

'pizza'

In [13]:
print(f"Similarity of king and man: {pretrained_w2v.similarity("king", "man")*100:.2f}%")
print(f"Similarity of king and woman: {pretrained_w2v.similarity("king", "woman")*100:.2f}%")

Similarity of king and man: 22.94%
Similarity of king and woman: 12.85%


In [14]:
import numpy as np

built_vector = pretrained_w2v["king"] - pretrained_w2v["man"] + pretrained_w2v["woman"]
print(
    "Similarity of king and our built vector:", 
    np.dot(
        pretrained_w2v["king"], 
        built_vector
    )*100/(
        np.linalg.norm(pretrained_w2v["king"])
        *np.linalg.norm(built_vector)
    )
)

Similarity of king and our built vector: 84.49392


In [16]:
for word in ["happy", "Happy", "HAPPY"]:
    print(f"How {word} is Tom? {pretrained_w2v.similarity(word, "Tom")*100:.2f}%")

How happy is Tom? 13.56%
How Happy is Tom? 5.92%
How HAPPY is Tom? 2.63%


In [20]:
counts = {
    "happy": pretrained_w2v.get_vecattr("happy", "count"),
    "Happy": pretrained_w2v.get_vecattr("Happy", "count"),
    "HAPPY": pretrained_w2v.get_vecattr("HAPPY", "count"),
}

total_count = sum(counts.values())

v_unified = sum(
    (count / total_count) * pretrained_w2v[word] for word, count in counts.items()
)

print(
    "Similarity of happy and Tom:", 
    np.dot(
        v_unified,
        pretrained_w2v["Tom"]
    )*100/(
        np.linalg.norm(v_unified)
        *np.linalg.norm(pretrained_w2v["Tom"])
    )
)
print("...via freq-weighted summation.")

Similarity of happy and Tom: 8.697032131728967
...via freq-weighted summation.


In [23]:
pivot = "happy"
variants = ["happy", "Happy", "HAPPY"]

sims = {
    word: pretrained_w2v.similarity(pivot, word) 
    for word in variants
}

total_sim = sum(sims.values())

v_unified = sum(
    (sim / total_sim) * pretrained_w2v[word] 
    for word, sim in sims.items()
)

dot_product = np.dot(v_unified, pretrained_w2v["Tom"])
norms = np.linalg.norm(v_unified) * np.linalg.norm(pretrained_w2v["Tom"])
similarity_pct = (dot_product / norms) * 100

print(f"Weights used: {sims}")
print(f"Similarity of unified 'happy' and 'Tom': {similarity_pct:.2f}%")
print("...via similarity-weighted summation.")

Weights used: {'happy': np.float32(1.0), 'Happy': np.float32(0.47028428), 'HAPPY': np.float32(0.29588777)}
Similarity of unified 'happy' and 'Tom': 11.36%
...via similarity-weighted summation.
